# Assemble cell type frequencies with pseudocount CLR

In this notebook, we read our scRNA-seq data and clinical labs, and use these to produce frequencies and proportions of cell types at each of our cell type labeling levels for each sample.

These results are combined with sample-specific metadata to enable downstream analysis and comparisons.

In addition to computing counts/frequency and fraction of total cells per sample, we use the Absolute Lymphocyte Counts from Clinical Blood Counts performed on the same blood draws to estimate absolute cell counts for each sample. This is done by computing the ratio between the lymphocyte counts in the scRNA-seq data and the ALC to get an Absolute count per RNA count correction factor, and then multiplying this ratio by the number of cells observed for each cell type:

$$C_{type} = N_{type} * {{ALC}\over N_{Tcells} + N_{Bcells} + N_{NKcells}}$$

$ALC$: Absolute lymphocyte count  
$C_{type}$: Estimated absolute count for a specific type  
$N_{type}$: count of cells of a specific type in scRNA-seq sample  
$N_{Tcells} + N_{Bcells} + N_{NKcells}$: count of lymphocytes in scRNA-seq sample

## Output format
Outputs include the following columns:  

**Sample Metadata columns**  
`cohort.cohortGuid`: Cohort ID (BR1 or BR2)  
`subject.subjectGuid`: Subject ID  
`subject.biologicalSex`: Subject Sex (Female or Male)  
`subject.cmv`: Subject CMV Status (Negative or Positive)  
`subject.bmi`: Subject BMI (integer)  
`subject.race`: Subject race  
`subject.ethnicity`: Subject ethnicity  
`subject.birthYear`: Subject Birth Year  
`subject.ageAtFirstDraw`: Subject Age at earliest blood draw in study  
`sample.sampleKitGuid`: Sample Kit ID  
`sample.visitName`: Sample Visit Name  
`sample.drawDate`: Sample Draw Date (Year-Month)  
`sample.subjectAgeAtDraw`: Subject age at time of draw, based on year of Draw Date and Birth Year  
`specimen.specimenGuid`: Specimen ID (pbmc_sample_id in .h5 files)  
  
**Frequency-related columns**  
(for AIFI_L1 as an example; AIFI_L1 is replaced with AIFI_L2 and AIFI_L3 for those levels)  

`AIFI_L1`: Cell Type assignment  
`AIFI_L1_count`: Count of cells within this sample with cell type assignment  
`total_cells`: Total cells within this sample  
`scrna.lymphocyte_count`: Sum of T, NK, and B cells  
`bc.lymphocyte_count`: Absolute Lymphocyte Count (ALC) from clinical Blood Counts (bc.)  
`alc_ratio`: ALC per scRNA Lymphocyte Count  
`AIFI_L1_frac_total`: Fraction of cells with cell type assignment divided by Total cells for this sample  
`AIFI_L1_alc`: ALC estimate for this cell type assignment  
`AIFI_L1_clr`: Centered Log Ratio computed using AIFI_L1_frac_total for all types within this sample  
`AIFI_L1_count_pseudo`: Count of cells within this sample with cell type assignment with addition of a pseudocount (+ 1)  
`AIFI_L1_clr_pseudo`: Centered Log Ratio computed using AIFI_L1_count_pseudo  

In [1]:
quiet_library <- function(...) { suppressPackageStartupMessages(library(...)) }

quiet_library(hise)
quiet_library(data.table)
quiet_library(dplyr)

Warning message:
“package ‘data.table’ was built under R version 4.4.3”


In [2]:
if(!dir.exists("output")) {
    dir.create("output")
}

CLR Transformation function from Mansi Singh

In [3]:
clr_transform <- function(x) {
  if (length(x) == 0) {
    return(NA)  # return NA for empty vectors
  }
  geom_mean <- exp(mean(log(x)))
  return(log(x / geom_mean))
}

## Retrieve cell types

In [4]:
type_uuid = '1a44252c-8cab-4c8f-92c9-d8f3af633790'
type_csv = cacheFiles(list(type_uuid))
type_df = read.csv(type_csv) %>%
  filter(AIFI_L3 != "HBB+ MAIT")

[1] "downloading fileID 1a44252c-8cab-4c8f-92c9-d8f3af633790"


In [5]:
l1_types <- unique(type_df$AIFI_L1)
l2_types <- unique(type_df$AIFI_L2)
l3_types <- unique(type_df$AIFI_L3)

## Retrieve clinical lab results

In [6]:
labs_uuid <- "bd618af8-99e2-45f8-bed8-655348d4cfeb"
res <- cacheFiles(list(labs_uuid))

[1] "downloading fileID bd618af8-99e2-45f8-bed8-655348d4cfeb"


In [7]:
labs <- read.csv(res)

In [8]:
alc <- labs %>%
  select(sample.sampleKitGuid, bc.lymphocyte_count)

## Retrieve labeled cell metadata

In [9]:
meta_uuid <- "4a4d94b0-3a15-4403-b0f4-fe22204741e4"
res <- cacheFiles(list(meta_uuid))

[1] "downloading fileID 4a4d94b0-3a15-4403-b0f4-fe22204741e4"


In [10]:
meta <- fread(res)
meta <- as.data.frame(meta)

In [11]:
nrow(meta)

[1] 13795227

In [12]:
sample_meta <- meta %>%
  select(starts_with("cohort"),
         starts_with("subject"),
         starts_with("sample"),
         starts_with("specimen")) %>%
  unique()

In [13]:
nrow(sample_meta)

[1] 868

In [14]:
sample_cols <- names(sample_meta)

## Compute lymphocyte counts per sample

In [15]:
l1_types <- unique(meta$AIFI_L1)
l1_types

[1] "B cell"          "T cell"          "NK cell"         "ILC"            
[5] "DC"              "Monocyte"        "Progenitor cell" "Erythrocyte"    
[9] "Platelet"

In [16]:
lc_l1_types <- c("B cell", "NK cell", "T cell")
sample_lc <- meta %>%
  group_by(sample.sampleKitGuid) %>%
  mutate(total_cells = n()) %>%
  filter(AIFI_L1 %in% lc_l1_types) %>%
  group_by(sample.sampleKitGuid, total_cells) %>%
  summarise(scrna.lymphocyte_count = n(),
            .groups = "keep")

In [17]:
sample_lc <- sample_lc %>%
  left_join(alc, by = "sample.sampleKitGuid")

In [18]:
sample_lc <- sample_lc %>%
  mutate(alc_ratio = bc.lymphocyte_count / scrna.lymphocyte_count)

In [19]:
head(sample_lc)

sample.sampleKitGuid,total_cells,scrna.lymphocyte_count,bc.lymphocyte_count,alc_ratio
<chr>,<int>,<int>,<int>,<dbl>
KT00001,18231,13903,1337,0.09616630
KT00002,17766,13888,2173,0.15646601
KT00003,18788,15012,1861,0.12396749
KT00004,16849,14648,1444,0.09858001
KT00006,17550,10503,1406,0.13386651
KT00007,16526,13906,1824,0.13116640


## Assemble cell type counts

### L1

In [20]:
l1_counts <- meta %>%
  # Count each type per sample
    group_by(sample.sampleKitGuid, AIFI_L1) %>%
    summarise(AIFI_L1_count = n(), .groups = "keep")

# Incorporate all types to include any zeros
l1 <- data.frame(
    sample.sampleKitGuid = rep(unique(l1_counts$sample.sampleKitGuid), each = length(l1_types)),
    AIFI_L1 = l1_types
)

l1_counts <- l1 %>%
  # If a type is missing, we have zero counts
    left_join(l1_counts) %>%
    mutate(AIFI_L1_count = ifelse(is.na(AIFI_L1_count), 0, AIFI_L1_count)) %>%
  # Add ALC and total sample counts for use below
    left_join(sample_lc, by = "sample.sampleKitGuid") %>%
  # Compute fractions of total cells per sample
    mutate(AIFI_L1_frac_total = AIFI_L1_count / total_cells) %>%
  # Compute LC estimate
    mutate(AIFI_L1_alc = AIFI_L1_count * alc_ratio) %>%
  # Regroup by sample and compute CLR for fractions
    group_by(sample.sampleKitGuid) %>%
    mutate(AIFI_L1_clr = clr_transform(AIFI_L1_count / total_cells)) %>%
  # CLR transform with pseudocounts
    mutate(AIFI_L1_count_pseudo = AIFI_L1_count + 1) %>%
    mutate(AIFI_L1_clr_pseudo = clr_transform(AIFI_L1_count_pseudo / sum(AIFI_L1_count_pseudo)))

Joining with `by = join_by(sample.sampleKitGuid, AIFI_L1)`


In [21]:
head(l1_counts)

sample.sampleKitGuid,AIFI_L1,AIFI_L1_count,total_cells,scrna.lymphocyte_count,bc.lymphocyte_count,alc_ratio,AIFI_L1_frac_total,AIFI_L1_alc,AIFI_L1_clr,AIFI_L1_count_pseudo,AIFI_L1_clr_pseudo
<chr>,<chr>,<dbl>,<int>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
KT00001,B cell,1781,18231,13903,1337,0.0961663,0.097690747,171.2721715,2.0880254,1782,2.05465324
KT00001,T cell,10920,18231,13903,1337,0.0961663,0.598979760,1050.1359419,3.9014464,10921,3.86760445
KT00001,NK cell,1202,18231,13903,1337,0.0961663,0.065931655,115.5918866,1.6948373,1203,1.66173535
KT00001,ILC,8,18231,13903,1337,0.0961663,0.000438813,0.7693304,-3.3174633,9,-3.23361379
KT00001,DC,246,18231,13903,1337,0.0961663,0.013493500,23.6569086,0.1084267,247,0.07854997
KT00001,Monocyte,4004,18231,13903,1337,0.0961663,0.219625912,385.0498454,2.8981443,4005,2.86446049


Add sample metadata and arrange columns

In [22]:
l1_counts <- l1_counts %>%
  left_join(sample_meta, by = "sample.sampleKitGuid") %>%
  select(one_of(sample_cols), everything())

In [23]:
head(l1_counts)

cohort.cohortGuid,subject.subjectGuid,subject.biologicalSex,subject.cmv,subject.bmi,subject.race,subject.ethnicity,subject.birthYear,subject.ageAtFirstDraw,sample.sampleKitGuid,⋯,AIFI_L1_count,total_cells,scrna.lymphocyte_count,bc.lymphocyte_count,alc_ratio,AIFI_L1_frac_total,AIFI_L1_alc,AIFI_L1_clr,AIFI_L1_count_pseudo,AIFI_L1_clr_pseudo
<chr>,<chr>,<chr>,<chr>,<dbl>,<chr>,<chr>,<int>,<int>,<chr>,⋯,<dbl>,<int>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
BR1,BR1001,Female,Negative,23,Caucasian,Non-Hispanic origin,1987,32,KT00001,⋯,1781,18231,13903,1337,0.0961663,0.097690747,171.2721715,2.0880254,1782,2.05465324
BR1,BR1001,Female,Negative,23,Caucasian,Non-Hispanic origin,1987,32,KT00001,⋯,10920,18231,13903,1337,0.0961663,0.598979760,1050.1359419,3.9014464,10921,3.86760445
BR1,BR1001,Female,Negative,23,Caucasian,Non-Hispanic origin,1987,32,KT00001,⋯,1202,18231,13903,1337,0.0961663,0.065931655,115.5918866,1.6948373,1203,1.66173535
BR1,BR1001,Female,Negative,23,Caucasian,Non-Hispanic origin,1987,32,KT00001,⋯,8,18231,13903,1337,0.0961663,0.000438813,0.7693304,-3.3174633,9,-3.23361379
BR1,BR1001,Female,Negative,23,Caucasian,Non-Hispanic origin,1987,32,KT00001,⋯,246,18231,13903,1337,0.0961663,0.013493500,23.6569086,0.1084267,247,0.07854997
BR1,BR1001,Female,Negative,23,Caucasian,Non-Hispanic origin,1987,32,KT00001,⋯,4004,18231,13903,1337,0.0961663,0.219625912,385.0498454,2.8981443,4005,2.86446049


Save counts

In [24]:
l1_file <- paste0("output/ra_prog_all_AIFI_L1_frequencies_",Sys.Date(),".csv")

In [25]:
write.csv(l1_counts, l1_file,
          row.names = FALSE, quote = FALSE)

### L2

In [26]:
l2_counts <- meta %>%
  # Count each type per sample
    group_by(sample.sampleKitGuid, AIFI_L2) %>%
    summarise(AIFI_L2_count = n(), .groups = "keep")

# Incorporate all types to include any zeros
l2 <- data.frame(
    sample.sampleKitGuid = rep(unique(l2_counts$sample.sampleKitGuid), each = length(l2_types)),
    AIFI_L2 = l2_types
)

l2_counts <- l2 %>%
  # If a type is missing, we have zero counts
    left_join(l2_counts) %>%
    mutate(AIFI_L2_count = ifelse(is.na(AIFI_L2_count), 0, AIFI_L2_count)) %>%
  # Add ALC and total sample counts for use below
    left_join(sample_lc, by = "sample.sampleKitGuid") %>%
  # Compute fractions of total cells per sample
    mutate(AIFI_L2_frac_total = AIFI_L2_count / total_cells) %>%
  # Compute LC estimate
    mutate(AIFI_L2_alc = AIFI_L2_count * alc_ratio) %>%
  # Regroup by sample and compute CLR for fractions
    group_by(sample.sampleKitGuid) %>%
    mutate(AIFI_L2_clr = clr_transform(AIFI_L2_count / total_cells)) %>%
  # CLR transform with pseudocounts
    mutate(AIFI_L2_count_pseudo = AIFI_L2_count + 1) %>%
    mutate(AIFI_L2_clr_pseudo = clr_transform(AIFI_L2_count_pseudo / sum(AIFI_L2_count_pseudo)))

Joining with `by = join_by(sample.sampleKitGuid, AIFI_L2)`


Add sample metadata and arrange columns

In [27]:
l2_counts <- l2_counts %>%
  left_join(sample_meta, by = "sample.sampleKitGuid") %>%
  select(one_of(sample_cols), everything())

In [28]:
head(l2_counts)

cohort.cohortGuid,subject.subjectGuid,subject.biologicalSex,subject.cmv,subject.bmi,subject.race,subject.ethnicity,subject.birthYear,subject.ageAtFirstDraw,sample.sampleKitGuid,⋯,AIFI_L2_count,total_cells,scrna.lymphocyte_count,bc.lymphocyte_count,alc_ratio,AIFI_L2_frac_total,AIFI_L2_alc,AIFI_L2_clr,AIFI_L2_count_pseudo,AIFI_L2_clr_pseudo
<chr>,<chr>,<chr>,<chr>,<dbl>,<chr>,<chr>,<int>,<int>,<chr>,⋯,<dbl>,<int>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
BR1,BR1001,Female,Negative,23,Caucasian,Non-Hispanic origin,1987,32,KT00001,⋯,97,18231,13903,1337,0.0961663,0.0053206078,9.3281306,-0.410271740,98,-0.4274579
BR1,BR1001,Female,Negative,23,Caucasian,Non-Hispanic origin,1987,32,KT00001,⋯,381,18231,13903,1337,0.0961663,0.0208984696,36.6393584,0.957816657,382,0.9329952
BR1,BR1001,Female,Negative,23,Caucasian,Non-Hispanic origin,1987,32,KT00001,⋯,1140,18231,13903,1337,0.0961663,0.0625308540,109.6295764,2.053800823,1141,2.0272349
BR1,BR1001,Female,Negative,23,Caucasian,Non-Hispanic origin,1987,32,KT00001,⋯,18,18231,13903,1337,0.0961663,0.0009873293,1.7309933,-2.094610961,19,-2.0679864
BR1,BR1001,Female,Negative,23,Caucasian,Non-Hispanic origin,1987,32,KT00001,⋯,145,18231,13903,1337,0.0961663,0.0079534858,13.9441128,-0.008248976,146,-0.0288188
BR1,BR1001,Female,Negative,23,Caucasian,Non-Hispanic origin,1987,32,KT00001,⋯,7,18231,13903,1337,0.0961663,0.0003839614,0.6731641,-3.039072570,8,-2.9329839


Save counts

In [29]:
l2_file <- paste0("output/diha_AIFI_L2_frequencies_",Sys.Date(),".csv")

In [30]:
write.csv(l2_counts, l2_file,
          row.names = FALSE, quote = FALSE)

### L3

In [31]:
l3_counts <- meta %>%
  # Count each type per sample
    group_by(sample.sampleKitGuid, AIFI_L3) %>%
    summarise(AIFI_L3_count = n(), .groups = "keep")

# Incorporate all types to include any zeros
l3 <- data.frame(
    sample.sampleKitGuid = rep(unique(l3_counts$sample.sampleKitGuid), each = length(l3_types)),
    AIFI_L3 = l3_types
)

l3_counts <- l3 %>%
  # If a type is missing, we have zero counts
    left_join(l3_counts) %>%
    mutate(AIFI_L3_count = ifelse(is.na(AIFI_L3_count), 0, AIFI_L3_count)) %>%
  # Add ALC and total sample counts for use below
    left_join(sample_lc, by = "sample.sampleKitGuid") %>%
  # Compute fractions of total cells per sample
    mutate(AIFI_L3_frac_total = AIFI_L3_count / total_cells) %>%
  # Compute LC estimate
    mutate(AIFI_L3_alc = AIFI_L3_count * alc_ratio) %>%
  # Regroup by sample and compute CLR for fractions
    group_by(sample.sampleKitGuid) %>%
    mutate(AIFI_L3_clr = clr_transform(AIFI_L3_count / total_cells)) %>%
  # CLR transform with pseudocounts
    mutate(AIFI_L3_count_pseudo = AIFI_L3_count + 1) %>%
    mutate(AIFI_L3_clr_pseudo = clr_transform(AIFI_L3_count_pseudo / sum(AIFI_L3_count_pseudo)))

Joining with `by = join_by(sample.sampleKitGuid, AIFI_L3)`


Add sample metadata and arrange columns

In [32]:
l3_counts <- l3_counts %>%
  left_join(sample_meta, by = "sample.sampleKitGuid") %>%
  select(one_of(sample_cols), everything())

In [33]:
names(l3_counts)

[1] "cohort.cohortGuid"       "subject.subjectGuid"    
 [3] "subject.biologicalSex"   "subject.cmv"            
 [5] "subject.bmi"             "subject.race"           
 [7] "subject.ethnicity"       "subject.birthYear"      
 [9] "subject.ageAtFirstDraw"  "sample.sampleKitGuid"   
[11] "sample.visitName"        "sample.drawDate"        
[13] "sample.subjectAgeAtDraw" "specimen.specimenGuid"  
[15] "AIFI_L3"                 "AIFI_L3_count"          
[17] "total_cells"             "scrna.lymphocyte_count" 
[19] "bc.lymphocyte_count"     "alc_ratio"              
[21] "AIFI_L3_frac_total"      "AIFI_L3_alc"            
[23] "AIFI_L3_clr"             "AIFI_L3_count_pseudo"   
[25] "AIFI_L3_clr_pseudo"

In [34]:
head(l3_counts)

cohort.cohortGuid,subject.subjectGuid,subject.biologicalSex,subject.cmv,subject.bmi,subject.race,subject.ethnicity,subject.birthYear,subject.ageAtFirstDraw,sample.sampleKitGuid,⋯,AIFI_L3_count,total_cells,scrna.lymphocyte_count,bc.lymphocyte_count,alc_ratio,AIFI_L3_frac_total,AIFI_L3_alc,AIFI_L3_clr,AIFI_L3_count_pseudo,AIFI_L3_clr_pseudo
<chr>,<chr>,<chr>,<chr>,<dbl>,<chr>,<chr>,<int>,<int>,<chr>,⋯,<dbl>,<int>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
BR1,BR1001,Female,Negative,23,Caucasian,Non-Hispanic origin,1987,32,KT00001,⋯,71,18231,13903,1337,0.0961663,0.0038944655,6.8278069,0.3324226,72,0.2767856
BR1,BR1001,Female,Negative,23,Caucasian,Non-Hispanic origin,1987,32,KT00001,⋯,26,18231,13903,1337,0.0961663,0.0014261423,2.5003237,-0.6721608,27,-0.7040436
BR1,BR1001,Female,Negative,23,Caucasian,Non-Hispanic origin,1987,32,KT00001,⋯,3,18231,13903,1337,0.0961663,0.0001645549,0.2884989,-2.8316450,4,-2.6135861
BR1,BR1001,Female,Negative,23,Caucasian,Non-Hispanic origin,1987,32,KT00001,⋯,15,18231,13903,1337,0.0961663,0.0008227744,1.4424944,-1.2222071,16,-1.2272918
BR1,BR1001,Female,Negative,23,Caucasian,Non-Hispanic origin,1987,32,KT00001,⋯,329,18231,13903,1337,0.0961663,0.0180461851,31.6387111,1.8658004,330,1.7992122
BR1,BR1001,Female,Negative,23,Caucasian,Non-Hispanic origin,1987,32,KT00001,⋯,10,18231,13903,1337,0.0961663,0.0005485163,0.9616630,-1.6276722,11,-1.6019852


Save counts

In [35]:
l3_file <- paste0("output/diha_AIFI_L3_frequencies_",Sys.Date(),".csv")

In [36]:
write.csv(l3_counts, l3_file,
          row.names = FALSE, quote = FALSE)

## Store results in HISE

In order to store the results in HISE, we'll need to cache these files to register them, and then we can upload the CSV file for later steps.

In [37]:
study_space_uuid <- "de025812-5e73-4b3c-9c3b-6d0eac412f2a"
title <- paste("DIHA Cell Type Frequency, ALC, and CLR with pseudocount", Sys.Date())

In [38]:
search_id = ids::proquint(n_words = 3)
search_id

[1] "salim-pafab-lupon"

In [39]:
in_list <- list(labs_uuid, meta_uuid)

In [40]:
out_list <- list(l1_file, l2_file, l3_file)

In [41]:
uploadFiles(
    files = out_list,
    studySpaceId = study_space_uuid,
    title = title,
    inputFileIds = in_list,
    destination = search_id,
    store = "project",
    doPrompt = FALSE
)

checking if conda env can compile...



[1] "/home/workspace/environment/minimal"
[1] "Cannot determine the current notebook."
[1] "1) /home/workspace/sound-life-scrna-analysis/03-data-assembly/15-R_assemble_type_frequencies.ipynb"
[1] "2) /home/workspace/sound-life-scrna-analysis/03-data-assembly/15a-R_assemble_type_frequencies_pseudocount.ipynb"
[1] "3) /home/workspace/data-apps-vis/datasets/ra_progression/18-R_assemble_ra_type_frequencies.ipynb"


Please select (1-3)  2


$Message
[1] "General Okay-ness"

$VisualizationId
[1] "00000000-0000-0000-0000-000000000000"

$AbstractionId
[1] "00000000-0000-0000-0000-000000000000"

$TraceId
[1] "668d02de-2ff2-45a3-8be5-a7a47982a6de"

$ProcessId
[1] "cf2fa569-3667-4382-a34b-3319e3c56797"

$WorkflowId
[1] "3416bdb1-0fb8-43ae-b420-d7cad89565d8"

$FileIds
$FileIds[[1]]
[1] "d7585f2b-b99e-4877-8081-f303b3c8b527"

$FileIds[[2]]
[1] "8239dc8c-8fb7-4003-9232-749aa3442e0e"

$FileIds[[3]]
[1] "33f57aa7-780e-4388-9b4b-f5ee856b58b3"

In [42]:
sessionInfo()

R version 4.4.1 (2024-06-14)
Platform: x86_64-conda-linux-gnu
Running under: Ubuntu 22.04.5 LTS

Matrix products: default
BLAS/LAPACK: /home/workspace/environment/minimal/lib/libopenblasp-r0.3.28.so;  LAPACK version 3.12.0

locale:
 [1] LC_CTYPE=C.UTF-8    LC_NUMERIC=C        LC_TIME=C          
 [4] LC_COLLATE=C        LC_MONETARY=C       LC_MESSAGES=C      
 [7] LC_PAPER=C          LC_NAME=C           LC_ADDRESS=C       
[10] LC_TELEPHONE=C      LC_MEASUREMENT=C    LC_IDENTIFICATION=C

time zone: America/Los_Angeles
tzcode source: system (glibc)

attached base packages:
[1] stats     graphics  grDevices utils     datasets  methods   base     

other attached packages:
[1] dplyr_1.1.4       data.table_1.17.2 hise_2.16.0      

loaded via a namespace (and not attached):
 [1] ids_1.0.1         crayon_1.5.3      vctrs_0.6.5       httr_1.4.7       
 [5] cli_3.6.3         rlang_1.1.6       stringi_1.8.7     purrr_1.0.4      
 [9] generics_0.1.4    assertthat_0.2.1  jsonlite_2.0.0    glue_1